# Pandas API on Spark (pyspark.pandas)
La **API de Pandas sobre Spark** permite trabajar con datos distribuidos usando una sintaxis similar a Pandas.
Esto combina la **facilidad de Pandas** con la **escalabilidad de Spark**.

## ✅ Características principales:
- Sintaxis similar a Pandas.
- Ejecución distribuida sobre Spark.
- Permite `.map()`, `.apply()`, `groupby()`, `merge()`, etc.

## ✅ Diferencias con DataFrames tradicionales de Spark:
| Operación | Spark DataFrame | Pandas API on Spark |
|-----------|-----------------|---------------------|
| Selección | `df.select('col')` | `psdf[['col']]` |
| Filtro    | `df.filter(col > 5)` | `psdf[psdf.col > 5]` |
| Nueva columna | `df.withColumn('x', col*2)` | `psdf['x'] = psdf.col * 2` |
| Función personalizada | UDF | `.apply()` |


In [8]:

import findspark
import pandas
findspark.init()
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType
from pyspark.sql import SparkSession
from pyspark.sql.types import (
    StructType, StructField, StringType, DoubleType, DateType
)
import pyspark.pandas as ps


jars = [
    "/var/snp-dwh/spark_jobs/jars/spark-excel_2.12-3.5.1_0.20.4.jar",
    "/var/snp-dwh/spark_jobs/jars/poi-4.1.2.jar",
    "/var/snp-dwh/spark_jobs/jars/poi-ooxml-4.1.2.jar",
    "/var/snp-dwh/spark_jobs/jars/xmlbeans-3.1.0.jar",
    "/var/snp-dwh/spark_jobs/jars/commons-math3-3.6.1.jar",
    "/var/snp-dwh/spark_jobs/jars/ooxml-schemas-1.4.jar",
    "/var/snp-dwh/spark_jobs/jars/postgresql-42.7.3.jar"
]

spark = SparkSession.builder \
    .appName("ExcelFallback") \
    .config("spark.master", "local[*]") \
    .config("spark.jars", ",".join(jars)) \
    .config("spark.driver.memory", "8g") \
    .config("spark.executor.memory", "8g") \
    .config("spark.memory.fraction", "0.8") \
    .getOrCreate()
spark.sparkContext.setCheckpointDir('/var/snp-dwh/localtest/cheakpoint/')
# Definición del schema exacto que me pasaste

schema = StructType([
    StructField("codigo_", StringType(), True),
    StructField("TIPO", StringType(), True),
    StructField("EJE", StringType(), True),
    StructField("NOMBRE_DEL_OBJETIVO", StringType(), True),
    StructField("NOMBRE_DE_LA_POLITICA", StringType(), True),
    StructField("META", StringType(), True),
    StructField("INDICADOR", StringType(), True),
    StructField("FUENTE_DE_INFORMACION", StringType(), True),
    StructField("GRUPO_DE_DESAGREGACION", StringType(), True),
    StructField("NIVEL_DE_DESAGREGACION", StringType(), True),
    StructField("CODIGO_GEOGRAFICO_DPA", StringType(), True),
    StructField("MES_ANIO", StringType(), True),
    StructField("FECHA", DateType(), True),
    StructField("ESTIMADOR", DoubleType(), True),
    StructField("ERROR_ESTANDAR", DoubleType(), True),
    StructField("LIMITE_INFERIOR", DoubleType(), True),
    StructField("LIMITE_SUPERIOR", DoubleType(), True),
    StructField("COEFICIENTE_DE_VARIACION", DoubleType(), True),
    StructField("NUMERADOR", DoubleType(), True),
    StructField("DENOMINADOR", DoubleType(), True),
    StructField("NOMBRE_DEL_EJE", StringType(), True),
    StructField("PERIODICIDAD_FICHA_METODOLOGICA", StringType(), True),
    StructField("FECHA_DE_TRANSFERENCIA_FICHA_METODOLOGICA", StringType(), True),
    StructField("DESAGREGACION_FICHA_METODOLOGICA", StringType(), True),
    StructField("PERIODICIDAD_DEL_DATO", StringType(), True)
])



# Ruta en HDFS (ajusta si cambias el archivo)
hdfs_path = "hdfs://192.168.1.144:8020/data/datos_pnd2425-13-05-2025_n.xlsx"

# Leer usando dataAddress para empezar desde la fila 3



# Simulación de datos (usando un pequeño dataset para ejemplo)
data = [
    ("A1", "Tipo1", "Eje1", "Objetivo1", "Politica1", "Meta1", "Ind1", "Fuente1", "Grupo1", "Nivel1", "DPA1", "2025-05", None, 120.0, 1.5, 100.0, 140.0, 0.1, 500.0, 1000.0, "EjeName1", "Mensual", "2025-05-01", "Desag1", "Mensual"),
    ("B2", "Tipo2", "Eje2", "Objetivo2", "Politica2", "Meta2", "Ind2", "Fuente2", "Grupo2", "Nivel2", "DPA2", "2025-06", None, 80.0, 2.0, 60.0, 100.0, 0.2, 400.0, 800.0, "EjeName2", "Trimestral", "2025-06-01", "Desag2", "Trimestral")
]
df = spark.read.format("com.crealytics.spark.excel") \
    .option("header", "true") \
    .option("dataAddress", "'Indicadores PND24-25'!A3") \
    .option("maxRowsInMemory", 500) \
    .schema(schema) \
    .load(hdfs_path)

# Convertir a Pandas API on Spark
psdf = df.pandas_api()
psdf.head(3)


,codigo_,TIPO,EJE,NOMBRE_DEL_OBJETIVO,NOMBRE_DE_LA_POLITICA,META,INDICADOR,FUENTE_DE_INFORMACION,GRUPO_DE_DESAGREGACION,NIVEL_DE_DESAGREGACION,CODIGO_GEOGRAFICO_DPA,MES_ANIO,FECHA,ESTIMADOR,ERROR_ESTANDAR,LIMITE_INFERIOR,LIMITE_SUPERIOR,COEFICIENTE_DE_VARIACION,NUMERADOR,DENOMINADOR,NOMBRE_DEL_EJE,PERIODICIDAD_FICHA_METODOLOGICA,FECHA_DE_TRANSFERENCIA_FICHA_METODOLOGICA,DESAGREGACION_FICHA_METODOLOGICA,PERIODICIDAD_DEL_DATO
0,1.1.1,P,SOCIAL,1. Mejorar las condiciones de vida de la pobla...,1.1 Contribuir a la reducción de la pobreza y ...,Reducir la tasa de pobreza extrema por ingreso...,Tasa de pobreza extrema por ingresos,Instituto Nacional de Estadística y Censos (IN...,Geográfico,Nacional,None,Dic_10,2010-01-01,0.1310,NaN,NaN,NaN,NaN,NaN,NaN,Porcentaje,Anual: Estimación puntual de diciembre\nSemest...,Transferencia estimación puntual a junio: hast...,"Nacional, urbano, rural\nSexo",Mensual
1,1.1.1,P,SOCIAL,1. Mejorar las condiciones de vida de la pobla...,1.1 Contribuir a la reducción de la pobreza y ...,Reducir la tasa de pobreza extrema por ingreso...,Tasa de pobreza extrema por ingresos,Instituto Nacional de Estadística y Censos (IN...,Geográfico,Nacional,None,Dic_11,2011-01-01,0.1161,NaN,NaN,NaN,NaN,NaN,NaN,Porcentaje,Anual: Estimación puntual de diciembre\nSemest...,Transferencia estimación puntual a junio: hast...,"Nacional, urbano, rural\nSexo",Mensual
2,1.1.1,P,SOCIAL,1. Mejorar las condiciones de vida de la pobla...,1.1 Contribuir a la reducción de la pobreza y ...,Reducir la tasa de pobreza extrema por ingreso...,Tasa de pobreza extrema por ingresos,Instituto Nacional de Estadística y Censos (IN...,Geográfico,Nacional,None,Dic_12,2012-01-01,0.1118,NaN,NaN,NaN,NaN,NaN,NaN,Porcentaje,Anual: Estimación puntual de diciembre\nSemest...,Transferencia estimación puntual a junio: hast...,"Nacional, urbano, rural\nSexo",Mensual


## 1. Ejemplos de `.map()` en Pandas API on Spark
La función `.map()` se aplica sobre Series (`psdf['col']`) y permite aplicar operaciones elemento por elemento.


In [9]:

# Ejemplo 1: duplicar valores
psdf['ESTIMADOR_X2'] = psdf['ESTIMADOR'].map(lambda x: x * 2 if x is not None else None)

# Ejemplo 2: clasificar valores
psdf['CLASE'] = psdf['ESTIMADOR'].map(lambda x: 'ALTO' if x and x > 100 else 'BAJO')


# Ejemplo 4: inicial de EJE
psdf['INICIAL_EJE'] = psdf['EJE'].map(lambda x: x[0] if x else '?')

# Ejemplo 5: concatenar codigo y eje
ps.set_option('compute.ops_on_diff_frames', True)

psdf['CODIGO_FULL'] = psdf.apply(lambda row: f"{row.codigo_}-{row.EJE}" if row.codigo_ and row.EJE else None, axis=1)

psdf[['ESTIMADOR','ESTIMADOR_X2','CLASE','CODIGO_FULL']].head(5)


/opt/spark/python/pyspark/pandas/utils.py:1016: PandasAPIOnSparkAdviceWarning: If the type hints is not specified for `apply`, it is expensive to infer the data type internally.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)
/opt/spark/python/lib/pyspark.zip/pyspark/pandas/__init__.py:50: UserWarning: 'PYARROW_IGNORE_TIMEZONE' environment variable was not set. It is required to set this environment variable to '1' in both driver and executor sides if you use pyarrow>=2.0.0. pandas-on-Spark will set it for you but it does not work if there is a Spark context already launched.
/opt/spark/python/lib/pyspark.zip/pyspark/pandas/__init__.py:50: UserWarning: 'PYARROW_IGNORE_TIMEZONE' environment variable was not set. It is required to set this environment variable to '1' in both driver and executor sides if you use pyarrow>=2.0.0. pandas-on-Spark will set it for you but it does not work if there is a Spark context already launched.
/opt/spark/python/lib/pyspark.zip/pyspark/pandas/__i

,ESTIMADOR,ESTIMADOR_X2,CLASE,CODIGO_FULL
0,0.1310,0.2620,BAJO,1.1.1-SOCIAL
1,0.1161,0.2322,BAJO,1.1.1-SOCIAL
3,0.0861,0.1722,BAJO,1.1.1-SOCIAL
2,0.1118,0.2236,BAJO,1.1.1-SOCIAL
4,0.0765,0.1530,BAJO,1.1.1-SOCIAL


## 2. Operaciones comunes en Pandas API on Spark
- **Filtrado**: igual que en Pandas, usando condiciones.
- **Agrupaciones**: `.groupby()` y agregaciones.
- **Ordenamientos**: `.sort_values()`.


In [10]:

# Filtrar registros con ESTIMADOR > 50
psdf_filtrado = psdf[psdf['ESTIMADOR'] > 50]

# Agrupar y calcular promedio
psdf_group = psdf.groupby('EJE')['ESTIMADOR'].mean()

# Ordenar por ESTIMADOR
psdf_sorted = psdf.sort_values('ESTIMADOR', ascending=False)

print(psdf_filtrado.head(3))
print(psdf_group.head(3))
print(psdf_sorted.head(3))


/opt/spark/python/pyspark/pandas/groupby.py:649: FutureWarning: Default value of `numeric_only` will be changed to `False` instead of `True` in 4.0.0.
  warnings.warn(
25/08/04 12:30:22 WARN AttachDistributedSequenceExec: clean up cached RDD(499) in AttachDistributedSequenceExec(2640)
/opt/spark/python/lib/pyspark.zip/pyspark/pandas/__init__.py:50: UserWarning: 'PYARROW_IGNORE_TIMEZONE' environment variable was not set. It is required to set this environment variable to '1' in both driver and executor sides if you use pyarrow>=2.0.0. pandas-on-Spark will set it for you but it does not work if there is a Spark context already launched.
/opt/spark/python/lib/pyspark.zip/pyspark/pandas/__init__.py:50: UserWarning: 'PYARROW_IGNORE_TIMEZONE' environment variable was not set. It is required to set this environment variable to '1' in both driver and executor sides if you use pyarrow>=2.0.0. pandas-on-Spark will set it for you but it does not work if there is a Spark context already launched.


     codigo_ TIPO     EJE                                                                                                                           NOMBRE_DEL_OBJETIVO                                                                                                                                                                                                    NOMBRE_DE_LA_POLITICA                                                                             META                                                     INDICADOR                                                                                                                                                                                                                                                               FUENTE_DE_INFORMACION GRUPO_DE_DESAGREGACION NIVEL_DE_DESAGREGACION CODIGO_GEOGRAFICO_DPA MES_ANIO       FECHA   ESTIMADOR  ERROR_ESTANDAR  LIMITE_INFERIOR  LIMITE_SUPERIOR  COEFICIENTE_DE_VARIACION  NUMERADOR  DENOMIN

25/08/04 12:30:25 WARN AttachDistributedSequenceExec: clean up cached RDD(529) in AttachDistributedSequenceExec(2885)
25/08/04 12:30:25 WARN AttachDistributedSequenceExec: clean up cached RDD(535) in AttachDistributedSequenceExec(2905)


EJE
INSTITUCIONAL         1.386304e+06
Gestión de riesgos    2.554664e+01
GESTIÓN DE RIESGOS    4.273964e+01
Name: ESTIMADOR, dtype: float64


25/08/04 12:30:29 WARN AttachDistributedSequenceExec: clean up cached RDD(557) in AttachDistributedSequenceExec(3172)


      codigo_ TIPO                                        EJE                                                                                                                           NOMBRE_DEL_OBJETIVO                                                                                                                                                                                                                                                                    NOMBRE_DE_LA_POLITICA                                                                                                                META                                                 INDICADOR                                                                                                                                                                                                                                                                                                                                    FUENTE_DE_INFORMAC

25/08/04 12:30:29 WARN AttachDistributedSequenceExec: clean up cached RDD(565) in AttachDistributedSequenceExec(3207)


## ✅ Conclusión
- La **API de Pandas sobre Spark** permite escribir código estilo Pandas, pero ejecutado de manera distribuida.
- `.map()` y `.apply()` funcionan igual que en Pandas, pero sobre grandes datasets.
- Es ideal para quienes conocen Pandas y quieren escalar sin reescribir lógica a DataFrames Spark.


In [11]:
spark.stop()